# Sintesis aditiva

Una senal periodica puede construirse sumando armonicos. Las amplitudes y fases forman el espectro: la receta de la senal.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Checkbox, Dropdown
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 4.5)
plt.rcParams["axes.grid"] = True


def setup_complex_axis(ax, lim=2, title=None):
    ax.axhline(0, color="0.55", lw=1)
    ax.axvline(0, color="0.55", lw=1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_xlabel("Real")
    ax.set_ylabel("Imaginaria")
    if title:
        ax.set_title(title)

def arrow(ax, z, color="C0", label=None, origin=0+0j, width=0.006):
    ax.arrow(origin.real, origin.imag, z.real, z.imag,
             head_width=0.08, length_includes_head=True,
             color=color, width=width, label=label)

def plot_phasor_sum(amplitudes, freqs, phases, t=0.0, decay=None, ncycles=2):
    amplitudes = np.asarray(amplitudes, dtype=float)
    freqs = np.asarray(freqs, dtype=float)
    phases = np.asarray(phases, dtype=float)
    if decay is None:
        decay = np.zeros_like(amplitudes)
    decay = np.asarray(decay, dtype=float)
    z = amplitudes * np.exp(-decay*t) * np.exp(1j*(freqs*t + phases))
    ts = np.linspace(0, ncycles*2*np.pi, 900)
    wave = np.sum(amplitudes[:, None] * np.exp(-decay[:, None]*ts)
                  * np.exp(1j*(freqs[:, None]*ts + phases[:, None])), axis=0)
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4))
    setup_complex_axis(ax0, max(1.2, 1.2*np.sum(np.abs(amplitudes))), "Fasores")
    current = 0+0j
    for comp in z:
        arrow(ax0, comp, origin=current, color="C1")
        current += comp
    ax0.plot(wave.real, wave.imag, color="0.75", lw=1)
    ax0.scatter([current.real], [current.imag], color="C3")
    ax1.plot(ts, wave.real, color="C0", label="Re")
    ax1.plot(ts, wave.imag, color="C2", alpha=0.7, label="Im")
    ax1.axvline(t, color="0.2", ls="--", lw=1)
    ax1.set_xlabel("t")
    ax1.legend()
    plt.show()


In [ ]:
def additive_synthesis(f0=220, window_ms=20,
                       A1=1.0, A2=0.0, A3=0.0, A4=0.0, A5=0.0, A6=0.0, A7=0.0):
    fs = 44100
    t = np.arange(0, 2.0, 1/fs)
    amps = np.array([A1, A2, A3, A4, A5, A6, A7])
    harmonics = np.arange(1, 8)
    components = amps[:, None] * np.sin(2*np.pi*(harmonics[:, None]*f0)*t)
    snd = components.sum(axis=0)
    n = int(fs*window_ms/1000)
    fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(11, 4))
    ax0.bar(harmonics, amps, color="C3")
    ax0.set_xlabel("armonico")
    ax0.set_ylabel("amplitud")
    denom = np.max(np.abs(snd)) or 1
    ax1.plot(1000*t[:n], snd[:n]/denom)
    for comp in components:
        ax1.plot(1000*t[:n], comp[:n]/denom, color="0.75", lw=0.8)
    ax1.set_xlabel("tiempo (ms)")
    ax1.set_title("forma de onda")
    plt.show()
    return snd, fs

if WIDGETS_AVAILABLE:
    interact(additive_synthesis, f0=(100, 440, 10), window_ms=(5, 100, 1),
             A1=(0.0, 1.0, 0.05), A2=(0.0, 1.0, 0.05), A3=(0.0, 1.0, 0.05),
             A4=(0.0, 1.0, 0.05), A5=(0.0, 1.0, 0.05), A6=(0.0, 1.0, 0.05), A7=(0.0, 1.0, 0.05))
else:
    additive_synthesis()
